In [1]:
import torch.optim as optim
import pickle
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
import wandb
import os
import numpy as np
import matplotlib.pyplot as plt
import torch._utils_internal
import  pandas as pd
# from ConvolutionalAutoencoder import ConvolutionalAutoencoder

In [2]:
def load_tensor_from_pickle(file_path):
    with open(file_path, 'rb') as f:
        loaded_tensor = pickle.load(f)
    return loaded_tensor

loaded_tensor = load_tensor_from_pickle("dataset/encoded_tensor.pickle")


In [3]:
loaded_tensor.shape

torch.Size([111000, 8, 6])

In [5]:
loaded_tensor[:1100]

tensor([[[ -2.0023,   9.4221,  16.8175,  -7.2667,   2.0052,  13.8680],
         [ -3.8738,  11.3990,  -5.2806, -11.2004,  -3.6282,  11.7889],
         [  8.4793,  13.5827,   8.7883,  11.8678,  -5.3639,  -0.3577],
         [  4.0678,  13.7938,   1.0248,  -9.0956,  -1.0353,   8.7844],
         [ -7.3203,  10.0596,  15.5890,   7.9606,  16.3409, -10.3198],
         [ -2.6853, -12.6293,  -8.7005,  -2.6771, -12.3955,   1.5797],
         [-16.4795,  10.7921, -12.6966,  14.4034,  13.4734,  -9.7041],
         [ -4.5330, -14.8162,   2.0145, -16.0187,   7.7566,   2.5373]],

        [[ -1.0144,  10.6092,  17.8839,  -7.8555,   1.2120,  14.1194],
         [ -3.1638,  11.2833,  -5.8544, -13.0960,  -6.3642,  14.6653],
         [  9.2745,  17.4473,  10.9124,  13.7114,  -6.9715,  -0.3679],
         [  3.1571,  17.1256,  -0.8419,  -8.5800,   0.4686,   7.9327],
         [ -7.3007,  11.0833,  15.1802,  10.3968,  18.1786, -10.3177],
         [ -4.9721, -14.0374, -10.1964,  -2.1729, -14.3471,   2.4485],
    

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
with open(f"dataset/appended_tensor_small.pickle", 'wb') as f:
    pickle.dump(loaded_tensor[:1100], f)

In [14]:
loaded_tensor = load_tensor_from_pickle(f"dataset/appended_tensor_small.pickle")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/encoded_tensor_small.pickle'

In [13]:
loaded_tensor.shape

torch.Size([1100, 8, 6])

In [9]:
data = loaded_tensor.view(loaded_tensor.shape[0], -1)

In [10]:
data.shape

torch.Size([111000, 48])

In [11]:
data = load_tensor_from_pickle("dataset/encoded_tensor.pickle")


# Preprocess the data
# Flatten each data point
data = data.view(data.shape[0], -1)




In [12]:
data

tensor([[ -2.0023,   9.4221,  16.8175,  ..., -16.0187,   7.7566,   2.5373],
        [ -1.0144,  10.6092,  17.8839,  ..., -16.3094,   9.7200,   2.2536],
        [  0.1239,   6.5417,  17.2458,  ..., -12.1577,   8.2604,   0.0522],
        ...,
        [ -1.1098,  10.9317,  18.1950,  ..., -16.4752,   9.9217,   2.6085],
        [ -0.6306,  10.8683,  18.4728,  ..., -16.2957,   8.9829,   3.0194],
        [ -0.4059,   7.5639,  14.0178,  ..., -11.9645,   7.4324,   1.4636]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [13]:
# Split into sequences of 11 (10 for input, 1 for target)
sequences = [data[i-11:i] for i in range(11, len(data))]

# Split sequences into inputs and targets
inputs = [seq[:-1] for seq in sequences]
targets = [seq[-1] for seq in sequences]

In [19]:
type(sequences)

list

In [ ]:
inputs = torch.stack(inputs)

In [17]:
sequences[0]

tensor([[-2.0023e+00,  9.4221e+00,  1.6818e+01, -7.2667e+00,  2.0052e+00,
          1.3868e+01, -3.8738e+00,  1.1399e+01, -5.2806e+00, -1.1200e+01,
         -3.6282e+00,  1.1789e+01,  8.4793e+00,  1.3583e+01,  8.7883e+00,
          1.1868e+01, -5.3639e+00, -3.5771e-01,  4.0678e+00,  1.3794e+01,
          1.0248e+00, -9.0956e+00, -1.0353e+00,  8.7844e+00, -7.3203e+00,
          1.0060e+01,  1.5589e+01,  7.9606e+00,  1.6341e+01, -1.0320e+01,
         -2.6853e+00, -1.2629e+01, -8.7005e+00, -2.6771e+00, -1.2395e+01,
          1.5797e+00, -1.6479e+01,  1.0792e+01, -1.2697e+01,  1.4403e+01,
          1.3473e+01, -9.7041e+00, -4.5330e+00, -1.4816e+01,  2.0145e+00,
         -1.6019e+01,  7.7566e+00,  2.5373e+00],
        [-1.0144e+00,  1.0609e+01,  1.7884e+01, -7.8555e+00,  1.2120e+00,
          1.4119e+01, -3.1638e+00,  1.1283e+01, -5.8544e+00, -1.3096e+01,
         -6.3642e+00,  1.4665e+01,  9.2745e+00,  1.7447e+01,  1.0912e+01,
          1.3711e+01, -6.9715e+00, -3.6786e-01,  3.1571e+00,  1

In [21]:
inputs[0]

tensor([[-2.0023e+00,  9.4221e+00,  1.6818e+01, -7.2667e+00,  2.0052e+00,
          1.3868e+01, -3.8738e+00,  1.1399e+01, -5.2806e+00, -1.1200e+01,
         -3.6282e+00,  1.1789e+01,  8.4793e+00,  1.3583e+01,  8.7883e+00,
          1.1868e+01, -5.3639e+00, -3.5771e-01,  4.0678e+00,  1.3794e+01,
          1.0248e+00, -9.0956e+00, -1.0353e+00,  8.7844e+00, -7.3203e+00,
          1.0060e+01,  1.5589e+01,  7.9606e+00,  1.6341e+01, -1.0320e+01,
         -2.6853e+00, -1.2629e+01, -8.7005e+00, -2.6771e+00, -1.2395e+01,
          1.5797e+00, -1.6479e+01,  1.0792e+01, -1.2697e+01,  1.4403e+01,
          1.3473e+01, -9.7041e+00, -4.5330e+00, -1.4816e+01,  2.0145e+00,
         -1.6019e+01,  7.7566e+00,  2.5373e+00],
        [-1.0144e+00,  1.0609e+01,  1.7884e+01, -7.8555e+00,  1.2120e+00,
          1.4119e+01, -3.1638e+00,  1.1283e+01, -5.8544e+00, -1.3096e+01,
         -6.3642e+00,  1.4665e+01,  9.2745e+00,  1.7447e+01,  1.0912e+01,
          1.3711e+01, -6.9715e+00, -3.6786e-01,  3.1571e+00,  1

In [22]:
inputs = torch.stack(inputs)

In [24]:
split_idx = int(len(inputs) * 0.8)  # % for training
train_inputs, val_inputs = inputs[:split_idx], inputs[split_idx:]
train_targets, val_targets = targets[:split_idx], targets[split_idx:]

In [25]:
train_inputs[0]

tensor([[-2.0023e+00,  9.4221e+00,  1.6818e+01, -7.2667e+00,  2.0052e+00,
          1.3868e+01, -3.8738e+00,  1.1399e+01, -5.2806e+00, -1.1200e+01,
         -3.6282e+00,  1.1789e+01,  8.4793e+00,  1.3583e+01,  8.7883e+00,
          1.1868e+01, -5.3639e+00, -3.5771e-01,  4.0678e+00,  1.3794e+01,
          1.0248e+00, -9.0956e+00, -1.0353e+00,  8.7844e+00, -7.3203e+00,
          1.0060e+01,  1.5589e+01,  7.9606e+00,  1.6341e+01, -1.0320e+01,
         -2.6853e+00, -1.2629e+01, -8.7005e+00, -2.6771e+00, -1.2395e+01,
          1.5797e+00, -1.6479e+01,  1.0792e+01, -1.2697e+01,  1.4403e+01,
          1.3473e+01, -9.7041e+00, -4.5330e+00, -1.4816e+01,  2.0145e+00,
         -1.6019e+01,  7.7566e+00,  2.5373e+00],
        [-1.0144e+00,  1.0609e+01,  1.7884e+01, -7.8555e+00,  1.2120e+00,
          1.4119e+01, -3.1638e+00,  1.1283e+01, -5.8544e+00, -1.3096e+01,
         -6.3642e+00,  1.4665e+01,  9.2745e+00,  1.7447e+01,  1.0912e+01,
          1.3711e+01, -6.9715e+00, -3.6786e-01,  3.1571e+00,  1

In [3]:
loaded_tensor[:5]

tensor([[[-2.0023e+00,  9.4221e+00,  1.6818e+01, -7.2667e+00,  2.0052e+00,
           1.3868e+01],
         [-3.8738e+00,  1.1399e+01, -5.2806e+00, -1.1200e+01, -3.6282e+00,
           1.1789e+01],
         [ 8.4793e+00,  1.3583e+01,  8.7883e+00,  1.1868e+01, -5.3639e+00,
          -3.5771e-01],
         [ 4.0678e+00,  1.3794e+01,  1.0248e+00, -9.0956e+00, -1.0353e+00,
           8.7844e+00],
         [-7.3203e+00,  1.0060e+01,  1.5589e+01,  7.9606e+00,  1.6341e+01,
          -1.0320e+01],
         [-2.6853e+00, -1.2629e+01, -8.7005e+00, -2.6771e+00, -1.2395e+01,
           1.5797e+00],
         [-1.6479e+01,  1.0792e+01, -1.2697e+01,  1.4403e+01,  1.3473e+01,
          -9.7041e+00],
         [-4.5330e+00, -1.4816e+01,  2.0145e+00, -1.6019e+01,  7.7566e+00,
           2.5373e+00]],

        [[-1.0144e+00,  1.0609e+01,  1.7884e+01, -7.8555e+00,  1.2120e+00,
           1.4119e+01],
         [-3.1638e+00,  1.1283e+01, -5.8544e+00, -1.3096e+01, -6.3642e+00,
           1.4665e+01],
        

In [4]:
loaded_tensor.shape

torch.Size([111000, 8, 6])

In [5]:
old_tensor = load_tensor_from_pickle("_data_train_autoencoder_flat.pickle")

In [7]:
old_tensor[0]

tensor([[ 0.4099, -0.0300,  0.1200,  0.4299, -0.0100,  0.1200,  0.4500,  0.0000,
          0.1100,  0.4600,  0.0100,  0.1000,  0.4700,  0.0300,  0.0800,  0.3799,
         -0.0200,  0.1200,  0.4099,  0.0000,  0.1300,  0.4299,  0.0200,  0.1200,
          0.4500,  0.0300,  0.1000,  0.4600,  0.0400,  0.0800,  0.3601, -0.0100,
          0.1200,  0.3899,  0.0200,  0.1200,  0.4199,  0.0300,  0.1200,  0.4399,
          0.0500,  0.1000,  0.4600,  0.0600,  0.0800,  0.3401,  0.0000,  0.1100,
          0.3701,  0.0300,  0.1200,  0.3999,  0.0500,  0.1100,  0.4299,  0.0700,
          0.1000,  0.4500,  0.0800,  0.0700,  0.3301,  0.0100,  0.1000,  0.3501,
          0.0300,  0.1100,  0.3799,  0.0500,  0.1100,  0.4099,  0.0800,  0.1000,
          0.4299,  0.1000,  0.0700,  0.3999, -0.0500,  0.1400,  0.4199, -0.0300,
          0.1300,  0.4399, -0.0100,  0.1300,  0.4600,  0.0100,  0.1100,  0.4700,
          0.0200,  0.0900,  0.3701, -0.0400,  0.1400,  0.3999, -0.0100,  0.1400,
          0.4299,  0.0100,  

In [8]:
loaded_tensor[0]

tensor([[ -2.0023,   9.4221,  16.8175,  -7.2667,   2.0052,  13.8680],
        [ -3.8738,  11.3990,  -5.2806, -11.2004,  -3.6282,  11.7889],
        [  8.4793,  13.5827,   8.7883,  11.8678,  -5.3639,  -0.3577],
        [  4.0678,  13.7938,   1.0248,  -9.0956,  -1.0353,   8.7844],
        [ -7.3203,  10.0596,  15.5890,   7.9606,  16.3409, -10.3198],
        [ -2.6853, -12.6293,  -8.7005,  -2.6771, -12.3955,   1.5797],
        [-16.4795,  10.7921, -12.6966,  14.4034,  13.4734,  -9.7041],
        [ -4.5330, -14.8162,   2.0145, -16.0187,   7.7566,   2.5373]],
       device='cuda:0', grad_fn=<SelectBackward0>)

In [2]:
from ConvolutionalAutoencoder import ConvolutionalAutoencoder
from CustomDataLoader import CustomDataset

In [3]:
data_path = r"/mnt/d/sources/cgan/playground/convolutional/_data_train_autoencoder_flat.pickle"
orig_data = CustomDataset(pickle.load(open(data_path, "rb")))

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(device)

cuda


In [5]:
model = ConvolutionalAutoencoder().to(device)

In [6]:
model_path = r"/mnt/d/sources/cgan/playground/convolutional/saved_models/checkpoint_300.pth"
model.load_state_dict(torch.load(model_path)["model_state_dict"])

<All keys matched successfully>

In [7]:
appended_tensor = torch.empty(0, 8, 6).float().to(device)

In [10]:
for i, data in enumerate(orig_data):
    reconstruction, encoded = model(data)
    appended_tensor = torch.cat((appended_tensor, encoded), dim=0)

TypeError: conv1d() received an invalid combination of arguments - got (DataLoader, Parameter, Parameter, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, tuple of ints padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!DataLoader!, !Parameter!, !Parameter!, !tuple!, !tuple!, !tuple!, int)
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, str padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!DataLoader!, !Parameter!, !Parameter!, !tuple!, !tuple!, !tuple!, int)


In [53]:
# data_loader = torch.utils.data.DataLoader(data)

https://pytorch.org/tutorials/recipes/recipes/save_load_across_devices.html


In [74]:
print(model.encoder)

Sequential(
  (0): Conv1d(3, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (1): ReLU()
  (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (4): ReLU()
  (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv1d(16, 8, kernel_size=(3,), stride=(1,), padding=(1,))
  (7): ReLU()
  (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Linear(in_features=120, out_features=48, bias=True)
)


In [75]:
# len(data_loader[0])

In [78]:
# Set the model to evaluation mode
model.eval()

# Encode your data using the pretrained encoder
input_data = data[0].float().to(device)
# encoded_data = model.encoder(input_data)
reconstruction, encoded = model(input_data)

# Use the encoded data for another algorithm
# Example: Print the encoded data
# print(encoded)

here


RuntimeError: mat1 and mat2 shapes cannot be multiplied (8x15 and 120x48)

In [21]:
data[0].float().to(device).shape

torch.Size([3, 125])

In [18]:
input_data = torch.tensor(data[0].float().to(device))
input_data

/tmp/ipykernel_184134/910591254.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_data = torch.tensor(data[0].float().to(device))


tensor([[ 0.4099, -0.0300,  0.1200,  0.4299, -0.0100,  0.1200,  0.4500,  0.0000,
          0.1100,  0.4600,  0.0100,  0.1000,  0.4700,  0.0300,  0.0800,  0.3799,
         -0.0200,  0.1200,  0.4099,  0.0000,  0.1300,  0.4299,  0.0200,  0.1200,
          0.4500,  0.0300,  0.1000,  0.4600,  0.0400,  0.0800,  0.3601, -0.0100,
          0.1200,  0.3899,  0.0200,  0.1200,  0.4199,  0.0300,  0.1200,  0.4399,
          0.0500,  0.1000,  0.4600,  0.0600,  0.0800,  0.3401,  0.0000,  0.1100,
          0.3701,  0.0300,  0.1200,  0.3999,  0.0500,  0.1100,  0.4299,  0.0700,
          0.1000,  0.4500,  0.0800,  0.0700,  0.3301,  0.0100,  0.1000,  0.3501,
          0.0300,  0.1100,  0.3799,  0.0500,  0.1100,  0.4099,  0.0800,  0.1000,
          0.4299,  0.1000,  0.0700,  0.3999, -0.0500,  0.1400,  0.4199, -0.0300,
          0.1300,  0.4399, -0.0100,  0.1300,  0.4600,  0.0100,  0.1100,  0.4700,
          0.0200,  0.0900,  0.3701, -0.0400,  0.1400,  0.3999, -0.0100,  0.1400,
          0.4299,  0.0100,  

In [18]:
from datetime import datetime

In [20]:
current_time = datetime.now().time().strftime("%H:%M:%S")
print("Current time:", current_time)

Current time: 12:45:30


In [ ]:
import time
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
import torch
import pickle
import torch.nn as nn
import math
from torch.optim.lr_scheduler import StepLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Define constants
seq_len = 48
epochs = 10
ninp = 48  # The dimension of your input feature
nhid = 128  # 200  # Dimension of the feedforward network model in nn.TransformerEncoder
nlayers = 2  # Number of nn.TransformerEncoderLayer in nn.TransformerEncoder
nhead = 2  # Number of heads in nn.MultiheadAttention models
dropout = 0.1  # Dropout value
lr = 0.001
log_interval = 50
batch_size = 480
scheduler_step = 1000
lr_gamma = 0.95


def load_tensor_from_pickle(file_path):
    with open(file_path, 'rb') as f:
        loaded_tensor = pickle.load(f)
    return loaded_tensor


data = load_tensor_from_pickle(r"/mnt/d/sources/cgan/playground/convolutional/dataset/encoded_tensor.pickle")
data = data.view(-1, 48)
print(data.shape)


class TensorDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        src = self.data[idx:idx + self.seq_len]
        tgt = self.data[idx + 1:idx + 1 + self.seq_len]
        return src, tgt


dataset = TensorDataset(data, seq_len)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)


class TransformerModel(nn.Module):

    def __init__(self, ninp, nhead, nhid, nlayers, dropout=0.5):
        super(TransformerModel, self).__init__()
        self.model_type = 'Transformer'
        encoder_layers = nn.TransformerEncoderLayer(ninp, nhead, nhid, dropout)

        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, nlayers)
        self.ninp = ninp

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, src_mask):
        src = src * math.sqrt(self.ninp)
        output = self.transformer_encoder(src, src_mask)
        return output


model = TransformerModel(ninp, nhead, nhid, nlayers, dropout).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=scheduler_step, gamma=lr_gamma)

# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0.
    running_loss = 0.
    start_time = time.time()
    print(f'start of epoch {epoch + 1} at {datetime.now().time().strftime("%H:%M:%S")}')
    src_mask = model.generate_square_subsequent_mask(seq_len).to(device)
    for batch, (src, tgt) in enumerate(dataloader):
        print("batch no ", batch, " len src", len(src), " len target", len(tgt))
        src = src.to(device)
        tgt = tgt.to(device)
        optimizer.zero_grad()

        if src.size(0) != seq_len:
            src_mask = model.generate_square_subsequent_mask(src.size(0)).to(device)
        output = model(src, src_mask)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        running_loss += loss.item()
        if batch % log_interval == 0 and batch > 0:
            cur_loss = total_loss / log_interval
            elapsed = time.time() - start_time
            print('| epoch {:3d} | {:5d}/{:5d} batches | '
                  'lr {:02.6f} | ms/batch {:5.2f} | '
                  'loss {:5.2f}'.format(
                epoch + 1, batch, len(data) // seq_len, scheduler.get_last_lr()[0],
                elapsed * 1000 / log_interval,
                cur_loss))
            total_loss = 0
            start_time = time.time()
    running_loss /= len(dataloader)
    print(f'End of epoch {epoch + 1}, Running loss {running_loss:.2f}')

print("===============================================")
print(f'End of training at {datetime.now().time().strftime("%H:%M:%S")}')


In [ ]:
seq_len = 48
seq_len_tgt = 24  # The sequence length for target data. Modify this as you need.

class TensorDataset(Dataset):
    def __init__(self, data, seq_len, seq_len_tgt):
        self.data = data
        self.seq_len = seq_len
        self.seq_len_tgt = seq_len_tgt

    def __len__(self):
        return len(self.data) - max(self.seq_len, self.seq_len_tgt)

    def __getitem__(self, idx):
        src = self.data[idx:idx + self.seq_len]
        tgt = self.data[idx + 1:idx + 1 + self.seq_len_tgt]
        return src, tgt

dataset = TensorDataset(data, seq_len, seq_len_tgt)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# In your training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0.
    running_loss = 0.
    start_time = time.time()
    print(f'start of epoch {epoch + 1} at {datetime.now().time().strftime("%H:%M:%S")}')
    src_mask = model.generate_square_subsequent_mask(seq_len).to(device)
    for batch, (src, tgt) in enumerate(dataloader):
        print("batch no ", batch, " len src", len(src), " len target", len(tgt))
        src = src.to(device)
        tgt = tgt.to(device)
        optimizer.zero_grad()

        if src.size(0) != seq_len:
            src_mask = model.generate_square_subsequent_mask(src.size(0)).to(device)
        output = model(src, src_mask)
        if output.shape[0] != tgt.shape[0]:
            output = output[:tgt.shape[0]]  # Ensure output and tgt have the same shape
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()
        # Rest of your training loop ...
